In [37]:
import io
import torch

from pathlib import Path
from pprint import pprint
from fairseq import checkpoint_utils, options, tasks
from contextlib import redirect_stdout, redirect_stderr
from utils import REPO_ROOT, preprocess_mask_predict_data, trace_mask_predict_iterations

In [ ]:
prep_result = preprocess_mask_predict_data(
    input_dir=REPO_ROOT / "input",
    output_dir=REPO_ROOT / "output",
    model_dir=REPO_ROOT / "checkpoints" / "maskPredict_en_de",
    source_lang="en",
    target_lang="de",
    run_name="en_de_demo",
    workers=10,
)

In [ ]:
trace_result = trace_mask_predict_iterations(
    data_bin_dir=prep_result["data_bin_dir"],
    output_dir=REPO_ROOT / "output",
    model_dir=REPO_ROOT / "checkpoints" / "maskPredict_en_de",
    source_lang="en",
    target_lang="de",
    subset="test",
    run_name="en_de_demo",
    decoding_iterations=10,
    length_beam=5,
    max_sentences=20,
)

for record in trace_result["records"]:
    print(f"[{record['id']}]")
    print("src:", record["source"])
    print("tgt:", record["target"])
    print("hyp:", record["hypothesis"])
    print("iterations:")
    for step in record["iterations"]:
        print(f"iter {step['iteration']:02d} (masked={step['masked_tokens']:>2}): {step['text']}")


[0]
src: The capital of France is Paris .
tgt: Die Hauptstadt Frankreichs ist Paris .
hyp: Die Hauptstadt Frankreichs ist Paris .
iterations:
iter 00 (masked= 0): Die Hauptstadt Frankreichs ist Paris .
iter 01 (masked= 6): Die Hauptstadt Frankreichs ist Paris .
iter 02 (masked= 5): Die Hauptstadt Frankreichs ist Paris .
iter 03 (masked= 4): Die Hauptstadt Frankreichs ist Paris .
iter 04 (masked= 4): Die Hauptstadt Frankreichs ist Paris .
iter 05 (masked= 3): Die Hauptstadt Frankreichs ist Paris .
iter 06 (masked= 2): Die Hauptstadt Frankreichs ist Paris .
iter 07 (masked= 2): Die Hauptstadt Frankreichs ist Paris .
iter 08 (masked= 1): Die Hauptstadt Frankreichs ist Paris .
iter 09 (masked= 0): Die Hauptstadt Frankreichs ist Paris .
[1]
src: The capital of Germany is Berlin .
tgt: Die Hauptstadt Deutschlands ist Berlin .
hyp: Hauptstadt Deutschlands ist Berlin .
iterations:
iter 00 (masked= 0): Hauptstadt Deutschlands ist Berlin .
iter 01 (masked= 5): Hauptstadt Deutschlands ist Berlin 

In [ ]:
import io
import warnings
from contextlib import redirect_stdout, redirect_stderr

def get_encoder_output(sentence: str, data_bin_dir=prep_result["data_bin_dir"], model_dir=REPO_ROOT / "checkpoints" / "maskPredict_en_de", source_lang: str = "en", target_lang: str = "de"):
    data_bin_dir = Path(data_bin_dir)
    checkpoint_path = Path(model_dir) / "checkpoint_best.pt"
    cli_args = [
        str(data_bin_dir),
        "--path", str(checkpoint_path),
        "--task", "translation_self",
        "--source-lang", source_lang,
        "--target-lang", target_lang,
        "--remove-bpe",
        "--max-sentences", "20",
        "--decoding-iterations", "10",
        "--decoding-strategy", "mask_predict",
        "--length-beam", "5",
        "--gen-subset", "test",
        "--cpu",
    ]

    parser = options.get_generation_parser()
    args = options.parse_args_and_arch(parser, input_args=cli_args)

    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()), warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="FairseqModel is deprecated, please use FairseqEncoderDecoderModel or BaseFairseqModel instead",
            category=UserWarning,
        )
        task = tasks.setup_task(args)
        models, _ = checkpoint_utils.load_model_ensemble(
            args.path.split(":"),
            arg_overrides=eval(args.model_overrides),
            task=task,
        )

    model = models[0].cpu()
    model.eval()

    src_tokens = task.source_dictionary.encode_line(sentence, add_if_not_exist=False).long().unsqueeze(0)
    src_lengths = torch.LongTensor([src_tokens.size(1)])

    with torch.no_grad():
        encoder_out = model.encoder(src_tokens=src_tokens, src_lengths=src_lengths)

    return {
        "sentence": sentence,
        "src_tokens": src_tokens,
        "src_lengths": src_lengths,
        "encoder_out": {
            "encoder_out": encoder_out["encoder_out"].detach().cpu(),
            "encoder_padding_mask": None if encoder_out["encoder_padding_mask"] is None else encoder_out["encoder_padding_mask"].detach().cpu(),
            "predicted_lengths": encoder_out["predicted_lengths"].detach().cpu(),
        },
    }


In [ ]:
encoder_example = get_encoder_output(
    "China is a large country in Asia , and the capital of the country is Beijing .",
)

print("sentence:", encoder_example["sentence"])
print("src_tokens shape:", tuple(encoder_example["src_tokens"].shape))
print("src_lengths:", encoder_example["src_lengths"].tolist())

sentence: China is a large country in Asia , and the capital of the country is Beijing .
src_tokens shape: (1, 18)
src_lengths: [18]
encoder_out: tensor(18, 1, 512)
encoder_padding_mask: None
predicted_lengths: tensor(1, 10000)


In [ ]:
patch_src_sentence = "The capital of France is Paris ."
patch_tgt_sentence = "The capital of Germany is Berlin ."

patch_src = get_encoder_output(patch_src_sentence)
patch_tgt = get_encoder_output(patch_tgt_sentence)


In [ ]:
print("patch src sentence:", patch_src["sentence"])
print("patch tgt sentence:", patch_tgt["sentence"])
print()
print("patch src encoder_out shape:", tuple(patch_src["encoder_out"]["encoder_out"].shape))
print("patch tgt encoder_out shape:", tuple(patch_tgt["encoder_out"]["encoder_out"].shape))
print("patch src predicted_lengths shape:", tuple(patch_src["encoder_out"]["predicted_lengths"].shape))
print("patch tgt predicted_lengths shape:", tuple(patch_tgt["encoder_out"]["predicted_lengths"].shape))
print()
print("patch src token ids:", patch_src["src_tokens"].tolist())
print("patch tgt token ids:", patch_tgt["src_tokens"].tolist())


patch src sentence: The capital of France is Paris .
patch tgt sentence: The capital of Germany is Berlin .

patch src encoder_out shape: (8, 1, 512)
patch tgt encoder_out shape: (8, 1, 512)
patch src predicted_lengths shape: (1, 10000)
patch tgt predicted_lengths shape: (1, 10000)

patch src token ids: [[30, 1924, 9, 1769, 16, 1925, 6, 2]]
patch tgt token ids: [[30, 1924, 9, 1013, 16, 1408, 6, 2]]
